# Matmul Kernel

### Design
- We spawn as many CTAs as we have C-tiles (bM, bN)
- Each CTA will loop-through all K-tiles (= K/bK) or 8K/64 if K = 64 i.e 128 tiles
- GMEM Latency is hidden by CTA-parallelism on same SM
   - Here, numCTiles = (M/bM * N/bN) = (8K * 8K/(128*256)) = 2048
   - Number of CTAs per 128 SMs = 2048, CTAs per SM = 16

### Is this enough CTAs to hide GMEM latency?
- HBM BW per device = 8 TBps or 8192/128 GBps per SM
- 64 GBps per SM
- Freq ~= 2 GHz or 2e9
- GMEM Latency ~= 1000 cycles = 1e3 * 0.5e-9 = 5e-7 secs
- Data inflight to hide latency = 64e9 bytes/secs * 5e-7 secs
- Data inflight reqd = 34 KB
- Each CTA brings in bMxbK * bNxbK tiles each iter = (128x64 + 256x64) x2B (bf16)
- Each CTA brings in 48 KB, hence a second CTA is enough to hide GMEM latency

### ILP in K-tiles
- Each K-tile can be software pipelined in two ways:
- Prefetch N K-tiles to hide GMEM latency further
- Accumulate 2-4 K-tiles unrolled together to increase ILP with limited CTAs
- This will increase register/SMEM pressure
- This is done by Triton intrinsically when lowering tl.dot()

### Kernel 1: Row Major Block Tiles
- Blocks (CTAs) are assigned in row major order across the tensor
- Important to note:
   - `a_ptrs`/`b_ptrs` are 2D tensors of addresses for the block
   - size of the block is blkM x blkN x blkK
   - mask requires `offsets_` comparison! Not addresses
   - K-tile progression requires all addresses in 2D tile to be moved forward

### Offsets/Pointers visualization
1. a_ptr =
```
< ----------- K -------------->
<   [ a0 a1 a2 a3 ....... aK ]
|   [ a0 a1 a2 a3 ....... aK ]
|   [ a0 a1 a2 a3 ....... aK ]
M   [ a0 a1 a2 a3 ....... aK ]
|   ....
|   [ a0 a1 a2 a3 ....... aK ]
|   [ a0 a1 a2 a3 ....... aK ]
>
```
2. offsets_am = pid_m * blkM + tl.arange(0, blkM) % M
```
< ------ pid_m * blkM elements ------ > [0 1 2 3 ......... blkM ]
```

3. offsets_k = tl.arange(0, blkK)
```
[0 1 2 3 4 5 ......... blkK ]
```

4. Now we need to create a 2D pointer array for our block which is K-major
Addresses of block:
```
[ 0               1                  2                   3                 ...                blkK ]
[ strideK      (strideK + 1)       (strideK + 2)       (strideK + 3)       ...     (strideK + blkK)]
[ 2 * strideK  (2 * strideK + 1)   (2 * strideK + 2)   (2 * strideK + 3)  ...  (2 * strideK + blkK)]
...
...
...
[ blkM * strideK  (blkM * strideK + 1)   (blkM * strideK + 2)   (blkM * strideK + 3)  ...  (blkM * strideK + blkK)]
```

5.
These pointers above can be summarized as:
```
a_ptrs = a_ptr + offsets_am[:,None] * A.stride(1) + offsets_k[None,:] * A.stride(0)
#                           ^ outer dim is m, K-major              ^ inner dim is k 
```

### CTA to warps lowering

##### Global Loads/Stores
- A single block is expressed as Atile[blkM, blkK], Btile[blkK, blkN], Ctile[blkM, blkN]
- One possible lowering by Triton (say blkK = 64, blkM = 128, blkN = 256)
- For loads/stores blkK is threads per warp in A/B and blkN is threads per warp in C
- For `blkK=64` we can load fp16x4 per thread and hence an entire `blkK` in one warp
- We will have `blkM` = 128 warps, of which 4 will schedule their loads at once
- In 32 cycles, all warps will have scheduled their loads
- With `blkN` = 256 warps, actual warps = `min(blkM, blkN)` = 128 warps
   - This will lead to 64 cycles to schedule all Btile loads
- The rest of the ~350-1000 cycle latency will be filled by other CTA-warps (other Ctiles)
- SMEM required for 1 CTA = 128x64x2B + 256x64x2B = 48KB
- RMEM required for 1 CTA (Hopper) = 128x256x4B = 1024KB
- Maximum number of CTAs resident at a time per SM = 5 (48x5 ~= 240KB fills up SMEM)

##### SMEM Loads/Stores
- `128/4 = 32` warp-loads for A and `256/4 = 64` warp-loads for B are enough to hide SMEM latency already

##### MMA Instructions
- If Triton lowers to FMA instructions, we have `128x256` FMA instructions to execute 
(assuming 64-wide vectorized FMA is allowed)
- With 128 warps (`min(blkM, blkN)`), we have 256 FMA instructions to execute
   - At an op-rate of 4 FMAs/cycle, we will need 64 cycles to issue all FMAs
   - This means we have 64 cycles to complete 1 FMA instruction without SW pipelining which is more than enough
- If Triton lowers to WGMMA/UMMA instructions, we will have 1 128x256x64 WGMMA/UMMA op per CTA
   - To cover the latency of that instruction, we can schedule another CTA's WGMMA/UMMA

##### A/B tile load masking
- We cannot compare absolute address as in `a_ptrs` we need to compare inner-dim's out of bounds
```
    mask = k + offsets_k[None,:] < K
    a_blk = tl.load(a_ptrs, mask=mask, other=0.0)
    mask = k + offsets_k[:,None] < K
    b_blk = tl.load(b_ptrs, mask=mask, other=0.0)
```

# Corrected Kernel Math

- Let's assume a block-tile size of `bM,bN,bK` = `(128,256,64)`
- Assume we launch `4 warps/CTA` = `128Ts`
- Assume we have `192KB` of SMEM space
- C-Tile sizes:
   - If `bN=256`, `128x256x4B` = `128KB` (max accumulator in RMEM size, 1KB per thread, 1 CTA per SM)
   - If `bN=128`, `128x128x4B` = `64KB` (will fit 2 CTAs per SM)

### MMA Instruction Pipelining
- For Ampere/Volta, we will have several `m16n16k8` MMA isntructions to keep the TensorCore busy
- For Hopper/Blackwell/Rubin, with `bK=64` and `mmaK=16` we have 4 MMA instructions per K-tile to hide TensorCore latency
- If we keep 2 accumulators alive for 2 separate C-tiles, we can have 8 MMA instructions to occupy TensorCore, but this will require 2 additional A/B tiles that will eat up SMEM. Tradeoff.
- We can work backwards to calculate how many cycles each `128x256x16` MMA instruction takes from NVIDIA's chip-level specs and calculate number of MMA instructions required to hide latency.


### MMA Instruction Latency and Issue Rate to saturate a GPU

For dense BF16 inputs with FP32 accumulation on Blackwell SM100, the largest relevant PTX TCGen05 MMA is:

```ptx
tcgen05.mma.cta_group::2.kind::f16
```

with descriptor fields selecting `atype=bf16`, `btype=bf16`, `dtype=f32`, dense, and shape:

```text
M = 256, N = 256, K = 16
```

The work per instruction is:

```text
FMAs  = 256 * 256 * 16 = 1,048,576
FLOPs = 2 * FMAs       = 2,097,152
```

For B200-class Blackwell, using dense BF16/FP16 peak of about `2.25 PFLOP/s` per GPU, and treating `cta_group::2` as a two-SM/CTA-pair operation:

```text
cycles = instruction_FLOPs * SM_count * clock / (2 * GPU_peak_FLOPs)
```

With `SM_count = 148` and effective Tensor Core clock around `1.855 GHz`:

```text
cycles ~= 2,097,152 * 148 * 1.855e9 / (2 * 2.25e15)
       ~= 128 cycles
```

So the useful number is:

```text
largest dense BF16->FP32 tcgen05.mma reciprocal throughput ~= 128 cycles
```

For latency hiding, use:

```text
independent_MMA_count = ceil(TensorCore_latency_cycles / 128)
```

Examples:

```text
256-cycle latency  -> 2 independent MMAs
512-cycle latency  -> 4 independent MMAs
768-cycle latency  -> 6 independent MMAs
1024-cycle latency -> 8 independent MMAs
```

The metric `TensorCore_latency_cycles` is telling us that in this many cycles we need to issue another MMA instruction to the Tensor Cores to fully utilize the Hardware.

Sources: NVIDIA [PTX ISA TCGen05 docs](https://docs.nvidia.com/cuda/parallel-thread-execution/index.html), NVIDIA [CUTLASS Blackwell SM100 GEMMs](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/blackwell_functionality.html), NVIDIA [Blackwell architecture technical brief](https://resources.nvidia.com/en-us-blackwell-architecture/blackwell-architecture-technical-brief).

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def gemm_kernel_rowMajorBlocking(
    A: torch.tensor, B: torch.tensor, C: torch.tensor,
    M: tl.int32, N: tl.int32, K: tl.int32,
    blkM: tl.constexpr, blkN: tl.constexpr, blkK: tl.constexpr,
):
  ### Locate Correct block ###
  # if we match bM, bN, bK to wgmma/umma atom dimensions, tl.dot() can lower to TensorCore instructions.
  # m0 [ <---- N ------ >]
  # m1 [ <---- N ------ >]
  # m2 [ <---- N ------ >]
  # m3 [ <---- N ------ >]
  pid = tl.program_id(0)
  num_pids_n = tl.cdiv(N, blkN)
  pid_m = pid // num_pids_n
  pid_n = pid % num_pids_n

  ### Pointers/Tensors ###
  a_ptr = A.data()
  b_ptr = B.data()
  c_ptr = C.data()
  A_stride_m = A.stride(0)
  A_stride_k = A.stride(1)
  B_stride_k = B.stride(0)
  B_stride_n = B.stride(1)
  C_stride_m = C.stride(0)
  C_stride_n = C.stride(1)
  M = A.shape(0)
  N = B.shape(1)
  K = B.shape(0)

  ### Block offsets ###
  # a-blk (dim0 offsets for MxK tensor)
  offsets_am = pid_m * blkM + tl.arange(0, blkM) % blkM
  # b-blk (dim1 offsets for KxM tensor)
  offsets_bn = pid_n * blkN + tl.arange(0, blkN) % blkN
  # c-blk dim0 offsets
  offsets_cm = pid_m * blkM + tl.arange(0, blkM) # masked later
  # c-blk dim1 offsets
  offsets_cn = pid_n * blkN + tl.arange(0, blkN) # masked later
  offsets_k = tl.arange(0, blkK)
  # blkM x blkK pointers for A-blk
  a_ptrs = a_ptr + offsets_am[:,None] * A_stride_m + offsets_k[None,:] * A_stride_k
  # blkK x blkN pointers for B-blk
  b_ptrs = b_ptr + offsets_k[:,None] * B_stride_k + offsets_bn[None,:] * B_stride_n
  # blkM x blkN pointers for C-blk (stationary)
  c_ptrs = c_ptr + offsets_cm[:,None] * C_stride_m + offsets_cn[None,:] * C_stride_n
  
  ### K-tile loop ###
  k = 0
  c_blk = tl.zeros((blkM, blkN), dtype=tl.float32)
  while (k < K):
    # Important to note that a_ptrs is a 2D tensor of addresses
    # Bounds checking is best doing using offsets!!
    mask = k + offsets_k[None,:] < K
    a_blk = tl.load(a_ptrs, mask=mask, other=0.0)
    mask = k + offsets_k[:,None] < K
    b_blk = tl.load(b_ptrs, mask=mask, other=0.0)
    c_blk = tl.dot(a_blk, b_blk, c_blk)
    a_ptrs += blkK * A_stride_k # 2D broadcast to all addresses
    b_ptrs += blkK * B_stride_k # 2D broadcast to all addresses
    k += blkK
  mask = (c_ptrs[:, None] < M) & (c_ptrs[None,:] < N)
  tl.store(c_ptrs, c_blk, mask=mask)





: 